# 🐍 Day 06a: LLD — Snake Game

---

## 🎯 What You'll Master Today
- Grid-based game design
- Snake movement with deque
- Collision detection
- Food generation
- Complete playable game loop

---

```
╔═══════════════════════════════════════════════════════════════════╗
║  SNAKE GAME — DESIGN OVERVIEW                                    ║
╠═══════════════════════════════════════════════════════════════════╣
║                                                                   ║
║  Grid (10x10):          Data Structures:                         ║
║  ┌──────────┐           • Snake body: deque of (row, col)       ║
║  │ . . . . .│           • Head: deque[0]                         ║
║  │ . S S . .│           • Food: (row, col)                       ║
║  │ . . S . .│           • Direction: UP/DOWN/LEFT/RIGHT         ║
║  │ . . S . .│                                                     ║
║  │ . . * . .│  * = food  Movement:                              ║
║  └──────────┘           • Add new head in direction              ║
║                          • If food → don't remove tail (grow)    ║
║  Collision:             • If no food → remove tail               ║
║  • Wall: head out of bounds                                      ║
║  • Self: head in body set                                        ║
║                                                                   ║
║  Class Design:                                                   ║
║  ┌──────────┐  has  ┌───────┐  on  ┌──────┐                    ║
║  │SnakeGame │──────►│ Snake │─────►│ Grid │                     ║
║  └──────────┘       └───────┘      └──────┘                     ║
║       │                                │                          ║
║       │ has                             │ has                     ║
║  ┌──────────┐                     ┌──────┐                       ║
║  │Direction │                     │ Food │                       ║
║  └──────────┘                     └──────┘                       ║
║                                                                   ║
╚═══════════════════════════════════════════════════════════════════╝
```

---

In [ ]:
# ============================================================
# 1. Snake Game — Complete Implementation
# ============================================================

from collections import deque
from enum import Enum
import random

class Direction(Enum):
    UP = (-1, 0)
    DOWN = (1, 0)
    LEFT = (0, -1)
    RIGHT = (0, 1)


class Snake:
    def __init__(self, start_row, start_col):
        self.body = deque([(start_row, start_col)])   # head at [0]
        self.body_set = {(start_row, start_col)}       # O(1) collision check
        self.direction = Direction.RIGHT
    
    @property
    def head(self):
        return self.body[0]
    
    def change_direction(self, new_dir: Direction):
        # Prevent 180° turn (can't go LEFT if going RIGHT)
        opposite = {
            Direction.UP: Direction.DOWN,
            Direction.DOWN: Direction.UP,
            Direction.LEFT: Direction.RIGHT,
            Direction.RIGHT: Direction.LEFT
        }
        if new_dir != opposite[self.direction]:
            self.direction = new_dir
    
    def next_head_position(self):
        dr, dc = self.direction.value
        return (self.head[0] + dr, self.head[1] + dc)
    
    def move(self, grow=False):
        new_head = self.next_head_position()
        self.body.appendleft(new_head)
        self.body_set.add(new_head)
        
        if not grow:
            tail = self.body.pop()
            self.body_set.remove(tail)
    
    def hits_self(self):
        # Check if next position collides with body
        next_pos = self.next_head_position()
        # Body set MINUS the tail (which will move if not growing)
        return next_pos in self.body_set


class SnakeGame:
    def __init__(self, rows=10, cols=10):
        self.rows = rows
        self.cols = cols
        self.snake = Snake(rows // 2, cols // 2)
        self.food = None
        self.score = 0
        self.game_over = False
        self._place_food()
    
    def _place_food(self):
        """Place food at random empty cell."""
        empty = []
        for r in range(self.rows):
            for c in range(self.cols):
                if (r, c) not in self.snake.body_set:
                    empty.append((r, c))
        if empty:
            self.food = random.choice(empty)
        else:
            self.food = None   # Snake fills entire board = WIN!
    
    def step(self, direction: Direction = None) -> int:
        """Advance game by one step. Returns score or -1 if game over."""
        if self.game_over:
            return -1
        
        if direction:
            self.snake.change_direction(direction)
        
        next_pos = self.snake.next_head_position()
        r, c = next_pos
        
        # Wall collision
        if r < 0 or r >= self.rows or c < 0 or c >= self.cols:
            self.game_over = True
            return -1
        
        # Self collision (check before move so tail is still there)
        # But tail will be removed unless growing, so check carefully
        will_grow = (next_pos == self.food)
        if not will_grow and next_pos in self.snake.body_set:
            # next_pos might be the current tail (which will move)
            if next_pos != self.snake.body[-1]:
                self.game_over = True
                return -1
        elif will_grow and next_pos in self.snake.body_set:
            # Growing → tail stays → definite collision
            self.game_over = True
            return -1
        
        # Move
        self.snake.move(grow=will_grow)
        
        if will_grow:
            self.score += 1
            self._place_food()
        
        return self.score
    
    def render(self):
        """ASCII render of the game board."""
        grid = [['.' for _ in range(self.cols)] for _ in range(self.rows)]
        
        for i, (r, c) in enumerate(self.snake.body):
            grid[r][c] = 'H' if i == 0 else 'S'
        
        if self.food:
            fr, fc = self.food
            grid[fr][fc] = '*'
        
        border = '+' + '-' * (self.cols * 2 - 1) + '+'
        print(border)
        for row in grid:
            print('|' + ' '.join(row) + '|')
        print(border)
        print(f"Score: {self.score}  |  Game Over: {self.game_over}")


print("✅ Snake game classes defined")

In [ ]:
# ============================================================
# 2. Demo — Simulated Game Play
# ============================================================

random.seed(42)
game = SnakeGame(rows=8, cols=8)
print("=== Initial Board ===")
game.render()

# Play a sequence of moves
moves = [
    Direction.RIGHT, Direction.RIGHT, Direction.DOWN,
    Direction.DOWN, Direction.LEFT, Direction.LEFT,
    Direction.UP, Direction.RIGHT, Direction.RIGHT,
    Direction.RIGHT, Direction.DOWN, Direction.DOWN,
]

for i, move in enumerate(moves):
    result = game.step(move)
    if result == -1:
        print(f"\n💀 Game Over at move {i+1}!")
        game.render()
        break

if not game.game_over:
    print(f"\n=== After {len(moves)} moves ===")
    game.render()

In [ ]:
# ============================================================
# 3. LeetCode-Style Snake Game (Simpler API)
# ============================================================
# This is the exact API from LC 353: Design Snake Game

class SnakeGameLC:
    """
    LeetCode 353 style.
    food = list of [row, col] in the order they appear.
    move(direction) → returns score or -1.
    """
    def __init__(self, width: int, height: int, food: list):
        self.width = width
        self.height = height
        self.food = deque(food)
        self.snake = deque([(0, 0)])   # start at top-left
        self.body_set = {(0, 0)}
        self.score = 0
        self.dirs = {'U': (-1,0), 'D': (1,0), 'L': (0,-1), 'R': (0,1)}
    
    def move(self, direction: str) -> int:
        dr, dc = self.dirs[direction]
        head_r, head_c = self.snake[0]
        nr, nc = head_r + dr, head_c + dc
        
        # Wall check
        if nr < 0 or nr >= self.height or nc < 0 or nc >= self.width:
            return -1
        
        # Eat food?
        if self.food and [nr, nc] == self.food[0]:
            self.food.popleft()
            self.score += 1
        else:
            # Remove tail
            tail = self.snake.pop()
            self.body_set.remove(tail)
        
        # Self collision (check AFTER tail removal)
        if (nr, nc) in self.body_set:
            return -1
        
        self.snake.appendleft((nr, nc))
        self.body_set.add((nr, nc))
        return self.score

# --- Demo ---
game = SnakeGameLC(3, 3, [[1,2], [0,1]])
moves = ['R', 'D', 'R', 'U', 'L', 'U']
for m in moves:
    result = game.move(m)
    snake_list = list(game.snake)
    print(f"move({m}) → score={result}, snake={snake_list}")
    if result == -1:
        break

In [ ]:
# ============================================================
# 4. Complexity & Design Decisions
# ============================================================

print("""
╔════════════════════════════════════════════════════════════════╗
║  DESIGN DECISIONS & COMPLEXITY                               ║
╠════════════════════════════════════════════════════════════════╣
║                                                                ║
║  Why deque for snake body?                                    ║
║  • appendleft(head) → O(1)                                   ║
║  • pop(tail) → O(1)                                          ║
║  • List would be O(n) for insert(0, head)                    ║
║                                                                ║
║  Why body_set alongside deque?                                ║
║  • Self-collision check → O(1) with set                      ║
║  • Without set → O(n) scan of body                           ║
║                                                                ║
║  ┌─────────────────────────────────────────────┐              ║
║  │ Operation         │ Time │ Why              │              ║
║  │ move()            │ O(1) │ deque + set      │              ║
║  │ collision check   │ O(1) │ set lookup       │              ║
║  │ food placement    │ O(NM)│ scan empty cells │              ║
║  │ render            │ O(NM)│ build grid       │              ║
║  │ space             │ O(S) │ S = snake length │              ║
║  └─────────────────────────────────────────────┘              ║
║                                                                ║
║  Extensions for interviews:                                   ║
║  • Multiple food types (different scores)                    ║
║  • Obstacles / walls inside the grid                         ║
║  • Speed increases as snake grows                            ║
║  • Multiplayer (2 snakes, collision = game over)             ║
║  • Wrap-around borders (snake exits left, enters right)      ║
║                                                                ║
╚════════════════════════════════════════════════════════════════╝
""")

---

## 📝 Interview Questions

| # | Question | Key Points |
|---|----------|-----------|
| 1 | Design Snake game | Deque for body, set for O(1) collision, direction enum |
| 2 | Why deque not list? | O(1) appendleft + pop vs O(n) insert(0) |
| 3 | How to check self-collision? | body_set → O(1). Check AFTER tail removal |
| 4 | How to place food? | Random empty cell. Can optimize with available set |
| 5 | Prevent 180° turn? | Store opposite map. Reject if new_dir == opposite[current] |

---

## ┌──────────────────────────────────────────────────────────┐
## │              📌 KEY TAKEAWAYS                            │
## ├──────────────────────────────────────────────────────────┤
## │                                                          │
## │  • Deque + Set = O(1) move + O(1) collision              │
## │  • Move: add new head, remove tail (unless eating)       │
## │  • Collision check: after tail removal, before head add  │
## │  • Direction validation: block 180° reversal             │
## │  • LC 353 pattern: food queue, return score per move     │
## │                                                          │
## └──────────────────────────────────────────────────────────┘

---

## ⏭️ Next Up: **Chapter 04 — HLD: URL Shortener**